# Assignment 1 — Training pipeline: French bakery daily revenue

**Decision supported:** the owner plans next week's staff rota from the
expected revenue of each day. The forecast is therefore made **7 days ahead**,
so every feature must be known at least 7 days before the target day.

**Target:** total daily revenue of the bakery (EUR), on days the bakery is open.

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_PATH = Path("data/daily_sales_french_bakery.csv")
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)


def show(fig):
    # Inline in Jupyter; closed (non-blocking) when main.py runs this as a script.
    if "ipykernel" in sys.modules:
        plt.show()
    else:
        plt.close(fig)

## 1. Load and aggregate to one row per day

In the raw file `y` is **revenue in EUR** per product and day
(`y / unit_price` gives an integer number of units in 98.4% of rows),
so summing `y` over products gives the bakery's daily revenue.

In [ ]:
raw = pd.read_csv(DATA_PATH, parse_dates=["ds"])
print(raw.shape, raw["ds"].min().date(), "->", raw["ds"].max().date())

daily = raw.groupby("ds")["y"].sum().rename("revenue").to_frame()
daily = daily.asfreq("D")  # one row per calendar day (none are missing, but this guarantees it)
print(daily.shape)
daily.head()

## 2. Closed days

Days with 0 EUR of total revenue are days the bakery was **closed** (mostly
Wednesdays in low season, plus holidays). Closing is the owner's decision,
known in advance, not a signal of low demand: on the Wednesdays it opened,
revenue was in line with other weekdays.

Decision: train and evaluate **only on open days**; for a day known to be
closed the forecast is 0 EUR. The model is later used to estimate the revenue
the bakery would have made on the days it closed.

In [ ]:
daily["closed"] = daily["revenue"] == 0
print("closed days:", daily["closed"].sum())
print(daily[daily["closed"]].index.day_name().value_counts())

open_days = daily[~daily["closed"]]
print("\nmean revenue on open days by weekday:")
print(open_days.groupby(open_days.index.day_name())["revenue"].mean().round())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(daily.index, daily["revenue"], lw=0.8)
ax.scatter(daily.index[daily["closed"]], daily["revenue"][daily["closed"]], color="red", s=10, label="closed")
ax.set_ylabel("EUR / day")
ax.set_title("Daily revenue")
ax.legend()
show(fig)

## 3. Features

**All features must be known 7 days before the target day** (the rota is
made one week ahead). Yesterday's revenue is therefore *not* allowed.

| Group | Features | Why |
|---|---|---|
| calendar | `dow_*`, `month_*` (one-hot) | weekends sell much more; strong summer peak (tourism) |
| `lag7` | same weekday one week earlier (last *open* one if that day was closed) | recent level of demand |
| `roll4` | mean of the last 4 open days on the same weekday | same idea, less noisy than a single week |
| holidays | `public_holiday`, `school_holiday` (Zone B) | days off change who shops and when |
| `weekend_summer` | weekend × July/August | a summer Sunday is busier than "Sunday effect + August effect" added up |
| weather *(candidate, rejected in 4.1)* | `temp_max_c`, `precip_mm`, `sunshine_h` | tourists and passers-by depend on the weather |

**External data** (downloaded once by `data/fetch_external_data.py` and committed):
public holidays and Zone B school holidays from the French government, daily
weather from Open-Meteo. The dataset does not say where the bakery is; its
regional products (Kouign Amann, Far Breton, Nantais, Guérandais…) point to the
Guérande / southern Brittany coast, which is the location assumed.

⚠️ **Weather caveat:** we use the *observed* weather of the target day. In
reality the owner would only have a 7-day-ahead *forecast*, which is less
accurate, so the gain from weather measured here is an upper bound.

In [ ]:
feat = daily.copy()

# Revenue history with closed days masked out, so a closure does not look like a 0-EUR day.
rev_open = feat["revenue"].where(~feat["closed"])
by_dow = rev_open.groupby(feat.index.dayofweek)
# Same weekday, at least 7 days back: shift by one week within each weekday, then carry the last open value forward.
feat["lag7"] = by_dow.transform(lambda s: s.shift(1).ffill())
# Mean of the last 4 open values on the same weekday (rolling mean skips the NaNs of closed days).
feat["roll4"] = by_dow.transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean().ffill())

dow = pd.get_dummies(feat.index.dayofweek, prefix="dow", drop_first=True, dtype=float)
month = pd.get_dummies(feat.index.month, prefix="month", drop_first=True, dtype=float)
dow.index = month.index = feat.index
feat = pd.concat([feat, dow, month], axis=1)

# Holidays
public = pd.to_datetime(pd.read_csv("data/public_holidays_fr.csv")["date"])
feat["public_holiday"] = feat.index.isin(public).astype(float)

school = pd.read_csv("data/school_holidays_zone_b.csv", parse_dates=["start_date", "end_date"])
feat["school_holiday"] = 0.0
for _, period in school.iterrows():  # each period covers [start_date, end_date)
    feat.loc[(feat.index >= period["start_date"]) & (feat.index < period["end_date"]), "school_holiday"] = 1.0

# Interaction: weekend during the summer peak
is_weekend = feat.index.dayofweek >= 5
is_summer = feat.index.month.isin([7, 8])
feat["weekend_summer"] = (is_weekend & is_summer).astype(float)

# Weather
weather = pd.read_csv("data/weather_guerande.csv", parse_dates=["date"]).set_index("date")
feat = feat.join(weather)

FEATURE_GROUPS = {
    "calendar": list(dow.columns) + list(month.columns),
    "lag7": ["lag7"],
    "roll4": ["roll4"],
    "holidays": ["public_holiday", "school_holiday"],
    "weekend_summer": ["weekend_summer"],
    "weather": ["temp_max_c", "precip_mm", "sunshine_h"],
}
ALL_CANDIDATES = [f for group in FEATURE_GROUPS.values() for f in group]
# Final model: every group except weather (see section 4.1 for why).
FEATURES = [f for name, group in FEATURE_GROUPS.items() if name != "weather" for f in group]

# Modelling rows: open days with a complete vector of *all* candidate features,
# so the ablation and the final model are evaluated on exactly the same days.
model_df = feat[~feat["closed"]].dropna(subset=ALL_CANDIDATES)
print(len(model_df), "rows,", len(FEATURES), "features")
feat[ALL_CANDIDATES].describe().T.round(2)

In [ ]:
# Quick look: does each new feature relate to revenue? (open days only)
o = feat[~feat["closed"]]
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
o.groupby("school_holiday")["revenue"].mean().plot.bar(ax=axes[0], title="school holiday")
o.groupby("public_holiday")["revenue"].mean().plot.bar(ax=axes[1], title="public holiday")
axes[2].scatter(o["temp_max_c"], o["revenue"], s=5); axes[2].set_title("max temperature (°C)")
axes[3].scatter(o["roll4"], o["revenue"], s=5); axes[3].set_title("roll4")
for ax in axes[:2]:
    ax.set_ylabel("mean EUR / day")
fig.tight_layout()
show(fig)

## 4. Time-based split

A random split would let the model see days *after* the ones it predicts.

| Set | Period | Use |
|---|---|---|
| train | Jan 2021 → Mar 2022 | fit |
| validation | Apr → May 2022 | choose regularisation / learning rate |
| test | Jun → Sep 2022 | final, untouched comparison |

In [ ]:
VAL_START, TEST_START = "2022-04-01", "2022-06-01"

def split_of(idx):
    return np.where(idx < VAL_START, "train", np.where(idx < TEST_START, "val", "test"))

model_df = model_df.assign(split=split_of(model_df.index))
print(model_df["split"].value_counts())

parts = {s: model_df[model_df["split"] == s] for s in ["train", "val", "test"]}

# Scaler fitted on train only, then applied to val/test (no information from the future).
scaler = StandardScaler().fit(parts["train"][FEATURES])
X = {s: scaler.transform(p[FEATURES]).astype(np.float32) for s, p in parts.items()}
y = {s: p["revenue"].to_numpy(np.float32) for s, p in parts.items()}

# The PyTorch models learn a standardised target for stable gradient descent;
# predictions are converted back to EUR.
y_mean, y_std = y["train"].mean(), y["train"].std()

### 4.1 Which feature groups actually help?

Add the groups one at a time and refit (sklearn ridge, `alpha` chosen on
validation each time). **Decisions are taken on the validation column**; the
test column is shown only for the report.

In [ ]:
def fit_ridge_on(cols, alphas=(0.0, 1.0, 10.0, 30.0, 100.0, 300.0, 1000.0)):
    sc = StandardScaler().fit(parts["train"][cols])
    Xs = {s: sc.transform(p[cols]) for s, p in parts.items()}
    best = min(alphas, key=lambda a: mean_absolute_error(y["val"], Ridge(alpha=a).fit(Xs["train"], y["train"]).predict(Xs["val"])))
    m = Ridge(alpha=best).fit(Xs["train"], y["train"])
    return {
        "alpha": best,
        "val_MAE": mean_absolute_error(y["val"], m.predict(Xs["val"])),
        "test_MAE": mean_absolute_error(y["test"], m.predict(Xs["test"])),
        "test_R2": r2_score(y["test"], m.predict(Xs["test"])),
    }

rows, cols = {}, []
for name, group in FEATURE_GROUPS.items():
    cols = cols + group
    rows["+ " + name] = fit_ridge_on(cols)
ablation = pd.DataFrame(rows).T
ablation.round(3)

**Decision: weather is left out of the final model.** Adding it makes the
*validation* error worse (≈195 → ≈201 EUR/day). It happens to improve the test
error, but choosing features by looking at the test set would make the test
result optimistic and no longer an honest estimate. Weather would also only be
available as a 7-day-ahead *forecast* in practice, not the observed values
used here, so its real value is even lower.

## 5. One model, three implementations

The same model everywhere: **linear regression with an L2 (ridge) penalty**,
minimising

$$\sum_i (y_i - \mathbf{w}^\top \mathbf{x}_i - b)^2 + \alpha \lVert \mathbf{w} \rVert^2$$

(the intercept `b` is not penalised). Dividing by *n*, the PyTorch versions
minimise `MSE + (alpha / n) * ||w||^2`, which has the same minimiser.

In [ ]:
def metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
        "under_forecast_share": float(np.mean(y_pred < y_true)),
    }

### 5.1 scikit-learn — choose `alpha` on validation

In [ ]:
ALPHAS = [0.0, 1.0, 10.0, 30.0, 100.0, 300.0, 1000.0]
val_mae = {}
for a in ALPHAS:
    m = Ridge(alpha=a).fit(X["train"], y["train"])
    val_mae[a] = mean_absolute_error(y["val"], m.predict(X["val"]))
print({a: round(v, 1) for a, v in val_mae.items()})

ALPHA = min(val_mae, key=val_mae.get)
print("chosen alpha:", ALPHA)
sk_model = Ridge(alpha=ALPHA).fit(X["train"], y["train"])

### 5.2 Manual PyTorch loop

Raw tensors, autograd for the gradients, and the gradient step written by hand.

In [ ]:
Xt = {s: torch.tensor(v) for s, v in X.items()}
yt_train = torch.tensor((y["train"] - y_mean) / y_std)
n_train, n_features = Xt["train"].shape
l2 = ALPHA / n_train  # same penalty as sklearn's alpha, per sample


def train_manual(lr, epochs=3000):
    torch.manual_seed(SEED)
    w = torch.zeros(n_features, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    losses = []
    for _ in range(epochs):
        pred = Xt["train"] @ w + b
        loss = ((pred - yt_train) ** 2).mean() + l2 * (w ** 2).sum()
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
        losses.append(loss.item())
    return w.detach(), b.detach(), losses


def predict_manual(w, b, Xs):
    return (Xs @ w + b).numpy() * y_std + y_mean

### 5.3 Standard PyTorch: `nn.Module` + `torch.optim`

In [ ]:
class LinearRegressor(torch.nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.linear = torch.nn.Linear(n_in, 1)

    def forward(self, x):
        return self.linear(x).squeeze(-1)


def train_module(lr, epochs=3000):
    torch.manual_seed(SEED)
    model = LinearRegressor(n_features)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = torch.nn.MSELoss()
    losses = []
    for _ in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(Xt["train"]), yt_train) + l2 * (model.linear.weight ** 2).sum()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return model, losses


def predict_module(model, Xs):
    with torch.no_grad():
        return model(Xs).numpy() * y_std + y_mean

### 5.4 Choose the learning rate

The learning rate is an *optimisation* setting, not part of the model: the
model (and its single best solution) is fixed by the features and `alpha`. So
the learning rate is chosen as the one that **converges best — lowest final
training loss** within the epoch budget. Validation MAE is printed too: a
learning rate that is too small stops before converging, which acts like extra
regularisation and can look slightly better on validation by chance, but then
the PyTorch models are no longer the same model as sklearn's.

In [ ]:
LRS = [0.003, 0.01, 0.03, 0.1]
lr_table = {}
for lr in LRS:
    w, b, losses = train_manual(lr)
    lr_table[lr] = {"final_train_loss": losses[-1], "val_MAE": mean_absolute_error(y["val"], predict_manual(w, b, Xt["val"]))}
lr_table = pd.DataFrame(lr_table).T
print(lr_table.round(6))
LR = lr_table["final_train_loss"].idxmin()
print("chosen lr:", LR)

w_manual, b_manual, loss_manual = train_manual(LR)
torch_model, loss_module = train_module(LR)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(loss_manual, label="manual loop")
ax.plot(loss_module, "--", label="nn.Module + optim")
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("train loss"); ax.legend()
show(fig)

## 6. Comparison on the test set

Baselines:
- **Naive mean**: the average train revenue every day.
- **Seasonal naive**: same weekday last week (`lag7`) — a strong, realistic baseline.

In [ ]:
preds = {}
for s in ["train", "val", "test"]:
    preds[s] = pd.DataFrame({
        "actual": y[s],
        "naive_mean": np.full_like(y[s], y["train"].mean()),
        "seasonal_naive": parts[s]["lag7"].to_numpy(),
        "sklearn": sk_model.predict(X[s]),
        "torch_manual": predict_manual(w_manual, b_manual, Xt[s]),
        "torch_module": predict_module(torch_model, Xt[s]),
    }, index=parts[s].index).assign(split=s)

MODELS = ["naive_mean", "seasonal_naive", "sklearn", "torch_manual", "torch_module"]
results = pd.DataFrame({m: metrics(preds["test"]["actual"], preds["test"][m]) for m in MODELS}).T
results.round(3)

In [ ]:
# Do the three implementations agree? Compare coefficients (in standardised-feature space, EUR units).
coefs = pd.DataFrame({
    "sklearn": sk_model.coef_,
    "torch_manual": w_manual.numpy() * y_std,
    "torch_module": torch_model.linear.weight.detach().numpy().ravel() * y_std,
}, index=FEATURES)
print("max |coef difference| vs sklearn (EUR):")
print((coefs.drop(columns="sklearn").sub(coefs["sklearn"], axis=0)).abs().max().round(3))
coefs.round(1)

## 7. What did the closed days cost?

Using the sklearn model, estimate the revenue each closed day would have made.

In [ ]:
closed_feat = feat[feat["closed"]].dropna(subset=FEATURES)
closed_est = pd.Series(sk_model.predict(scaler.transform(closed_feat[FEATURES])), index=closed_feat.index, name="estimated_revenue")
print(f"{len(closed_est)} closed days with features, estimated lost revenue: {closed_est.sum():,.0f} EUR")
closed_est.round().to_frame().assign(weekday=closed_est.index.day_name())

## 8. Save artifacts for the dashboard (the app never retrains)

In [ ]:
joblib.dump(sk_model, ARTIFACTS / "sklearn_ridge.joblib")
joblib.dump(scaler, ARTIFACTS / "scaler.joblib")
torch.save({"w": w_manual, "b": b_manual}, ARTIFACTS / "torch_manual.pt")
torch.save(torch_model.state_dict(), ARTIFACTS / "torch_module.pt")

pd.concat(preds.values()).rename_axis("date").to_csv(ARTIFACTS / "predictions.csv")
feat.rename_axis("date").to_csv(ARTIFACTS / "features.csv")
closed_est.rename_axis("date").to_csv(ARTIFACTS / "closed_days_estimate.csv")

meta = {
    "features": FEATURES,
    "alpha": ALPHA,
    "lr": LR,
    "y_mean": float(y_mean),
    "y_std": float(y_std),
    "val_start": VAL_START,
    "test_start": TEST_START,
    "test_metrics": results.round(4).to_dict(orient="index"),
    "ablation": ablation.round(4).to_dict(orient="index"),
}
(ARTIFACTS / "meta.json").write_text(json.dumps(meta, indent=2))
print("saved:", sorted(p.name for p in ARTIFACTS.iterdir()))